# Assignment 2: Process Numerical Data / Grading Rubric

**Total: 10 points** (Tasks 1–9: 1 point each; Discussion: 1 point)

## Reference setup

Run the cells in order to reproduce the expected results.

In [12]:
import numpy as np

data = np.array([
    [101, 72.4, 2.1, 101.3, 12.5],
    [102, 85.1, 5.8, 98.7, 15.2],
    [103, 69.8, 1.9, 102.1, 11.8],
    [104, 91.3, 7.2, 97.4, 18.1],
    [105, 76.5, 3.0, 100.5, 13.7],
    [106, -999, 4.1, 99.9, 14.8],
    [107, 88.7, 6.5, 96.8, 17.3],
    [108, 73.2, 2.4, 101.8, 12.9],
    [109, 79.6, 3.8, 100.1, 14.2],
    [110, 95.2, 8.1, 95.9, 20.4]
])

## Task 1 — Inspect the batch (1.0 point)

Full credit requires `shape == (10, 5)`, `ndim == 2`, `size == 50`, and a floating-point dtype (normally `float64`). The explanation must state that a NumPy array has one common dtype; because measurement columns contain decimals, NumPy promotes all entries—including Machine ID—to floating point.

**Partial credit:** 0.5 for correct metadata without an adequate explanation, or for a correct explanation with materially incomplete metadata.

In [13]:
print('shape:', data.shape)
print('ndim:', data.ndim)
print('size:', data.size)
print('dtype:', data.dtype)

shape: (10, 5)
ndim: 2
size: 50
dtype: float64


## Task 2 — Separate identifiers from measurements (1.0 point)

Full credit requires a one-dimensional `machine_ids` array from column 0 with shape `(10,)` and a two-dimensional `measurements` array from columns 1–4 with shape `(10, 4)`. The student must confirm both shapes.

**Partial credit:** 0.5 if the values are correct but dimensionality is wrong (for example, IDs have shape `(10, 1)`), or if the split is correct but shapes are not confirmed.

In [14]:
machine_ids = data[:, 0]
measurements = data[:, 1:]
print(machine_ids.shape, measurements.shape)

(10,) (10, 4)


## Task 3 — Validate the records (1.0 point)

Full credit requires an elementwise comparison to `-999` reduced across measurement columns (`axis=1`) to create one Boolean value per row. The mask must be `[False, False, False, False, False, True, False, False, False, False]`, and the reported invalid Machine ID is `106`. No explicit Python loop.

**Partial credit:** 0.5 for detecting the invalid value but producing a mask of the wrong shape or failing to report the associated ID.

In [49]:
invalid_mask1 = np.any(measurements == -999, axis=1)
invalid_mask1

array([False, False, False, False, False,  True, False, False, False,
       False])

In [50]:


invalid_mask2 = np.all(measurements != -999, axis=1)
invalid_mask2


array([ True,  True,  True,  True,  True, False,  True,  True,  True,
        True])

In [46]:
invalid_mask = np.any(measurements == -999, axis=1)
invalid_mask

invalid_ids = machine_ids[invalid_mask]
print('Invalid mask:', invalid_mask)
print('Invalid IDs:', invalid_ids)

Invalid mask: [False False False False False  True False False False False]
Invalid IDs: [106.]


## Task 4 — Filter the batch (1.0 point)

Full credit requires Boolean filtering with the inverse validation mask, producing `clean_ids.shape == (9,)` and `clean_measurements.shape == (9, 4)`. Machine `106` must be absent, and row alignment must be preserved. No explicit Python loop.

**Partial credit:** 0.5 if only one array is filtered correctly or the two arrays become misaligned.

In [37]:
clean_ids = machine_ids[~invalid_mask]
clean_measurements = measurements[~invalid_mask]
print(clean_ids)
print(clean_ids.shape, clean_measurements.shape)

[101. 102. 103. 104. 105. 107. 108. 109. 110.]
(9,) (9, 4)


## Task 5 — Aggregate the clean data (1.0 point)

Full credit requires column-wise aggregation (`axis=0`) over clean data only. Expected results in the order temperature, vibration, pressure, energy are:

- Mean: `[81.31111111, 4.53333333, 99.4, 15.12222222]`
- Minimum: `[69.8, 1.9, 95.9, 11.8]`
- Maximum: `[95.2, 8.1, 102.1, 20.4]`

No explicit Python loop. **Partial credit:** approximately 0.33 per correct aggregation; cap at 0.5 if aggregation includes the rejected row or uses the wrong axis.

In [38]:
column_means = clean_measurements.mean(axis=0)
column_mins = clean_measurements.min(axis=0)
column_maxs = clean_measurements.max(axis=0)
print('Means:', column_means)
print('Minimums:', column_mins)
print('Maximums:', column_maxs)

Means: [81.31111111  4.53333333 99.4        15.12222222]
Minimums: [69.8  1.9 95.9 11.8]
Maximums: [ 95.2   8.1 102.1  20.4]


## Task 6 — Transform temperature (1.0 point)

Full credit requires a vectorized conversion into a new array using `F = C * 9/5 + 32`, while leaving the Celsius column unchanged. Expected Fahrenheit values: `[162.32, 185.18, 157.64, 196.34, 169.70, 191.66, 163.76, 175.28, 203.36]`. No explicit Python loop.

**Partial credit:** 0.5 for a correct formula that overwrites Celsius, or for an arithmetic/order error with otherwise correct vectorization.

In [39]:
temperatures_c = clean_measurements[:, 0]
temperatures_f = temperatures_c * 9 / 5 + 32
print(temperatures_f)

[162.32 185.18 157.64 196.34 169.7  191.66 163.76 175.28 203.36]


## Task 7 — Apply a quality rule (1.0 point)

Full credit requires two vectorized Boolean conditions combined with elementwise OR (`|`) and parentheses. Expected flagged Machine IDs: `[102, 104, 107, 110]`. Notice that `102` qualifies because temperature is greater than 85 even though vibration is not greater than 6. No explicit Python loop.

**Partial credit:** 0.5 for using AND instead of OR, applying one condition only, or constructing the mask correctly but not reporting the IDs.

In [40]:
inspection_mask = (clean_measurements[:, 0] > 85) | (clean_measurements[:, 1] > 6.0)
inspection_ids = clean_ids[inspection_mask]
print(inspection_ids)

[102. 104. 107. 110.]


## Task 8 — Center the measurements (1.0 point)

Full credit requires subtracting the four column means from the `(9, 4)` clean array by broadcasting, verifying with `np.allclose` (or an equivalent NumPy check) that the centered column means are approximately zero, and explaining floating-point roundoff. No explicit Python loop.

Typical residual means are on the order of `1e-15`; exact residual values may vary. **Partial credit:** 0.5 if centering is correct but verification or explanation is missing.

In [41]:
centered_measurements = clean_measurements - column_means
centered_means = centered_measurements.mean(axis=0)
print('Centered column means:', centered_means)
print('Approximately zero:', np.allclose(centered_means, 0))

Centered column means: [-6.31593543e-15  4.93432455e-17  7.89491929e-15  3.94745964e-16]
Approximately zero: True


## Task 9 — Produce a quality summary (1.0 point)

Full credit requires a final report showing 10 raw records, 9 clean records, 1 rejected record, and 4 flagged machines, plus a programmatic consistency check that clean + rejected equals raw. The check may be an `assert` or a printed Boolean. No explicit Python loop.

**Partial credit:** 0.5 for correct counts without a programmatic check, or a valid check with incomplete/incorrect reporting.

In [42]:
raw_count = data.shape[0]
clean_count = clean_measurements.shape[0]
rejected_count = invalid_mask.sum()
flagged_count = inspection_mask.sum()

print('QUALITY SUMMARY')
print('Raw records:', raw_count)
print('Clean records:', clean_count)
print('Rejected records:', rejected_count)
print('Machines flagged for inspection:', flagged_count)
assert clean_count + rejected_count == raw_count
print('Count consistency check: passed')

QUALITY SUMMARY
Raw records: 10
Clean records: 9
Rejected records: 1
Machines flagged for inspection: 4
Count consistency check: passed


## Exit Discussion — From 10 rows to 10 billion (1.0 point)

Full credit requires 2–3 sentences that identify at least two genuine complications—such as memory limits, partitioning, parallel execution, data movement, failure recovery, or avoiding full rescans—and explain which pipeline concepts remain the same.

**Model response:** Ten billion observations will not fit comfortably in one machine's memory, so the batch must be partitioned and processed in parallel while controlling data movement and recovering safely from worker failures. Validation masks, filtering rules, column transformations, and aggregations remain conceptually the same, but they must be executed per partition and combined with distributed reductions rather than evaluated in one in-memory NumPy array.

**Partial credit:** 0.5 if two complications are identified but conceptual continuity is missing, or vice versa.

## Optional Early-Finishers reference (ungraded)

Expected scores, in clean Machine ID order, are `[41.51, 59.04, 39.52, 67.17, 46.45, 63.63, 42.93, 50.34, 72.58]`. Machine `110` has the highest score. A strong explanation notes that the weighted sum is scale-dependent: a feature with a larger numerical range can dominate unless the variables are standardized/normalized or the weights are calibrated to units and domain meaning.

In [43]:
health_scores = (
    0.4 * clean_measurements[:, 0]
    + 3 * clean_measurements[:, 1]
    + 0.5 * clean_measurements[:, 3]
)
highest_score_id = clean_ids[np.argmax(health_scores)]
print('Scores:', health_scores)
print('Highest-score Machine ID:', highest_score_id)

Scores: [41.51 59.04 39.52 67.17 46.45 63.63 42.93 50.34 72.58]
Highest-score Machine ID: 110.0


## Grading checks

- The submission is one notebook with code, visible printed results, and short written responses.
- The organization is easy to follow: raw batch → validation → filtering → transformation → aggregation → quality summary. Minor differences in section order should not lose credit if the workflow remains clear.
- Tasks 3–9 contain no explicit Python `for` loops. If a loop replaces the requested NumPy operation, deduct within the affected task; do not impose an additional global penalty.
- Equivalent NumPy expressions and variable names are acceptable when they meet the stated behavior.
- Machine IDs may print with `.0` because the source array is floating point; this is not an error.
- Use reasonable tolerance for floating-point results (for example, `np.allclose`).

## Grader notes

**Score:** ______ / 10

**Feedback:**
- Strengths:
- Corrections needed:
- Suggested next step: